# PhyDG-Slide 2/2: train (GPU, runs in the background)

1. **Add Input -> Your Work ->** `kaggle_1_prepare` (provides `cache.tar`).
2. Settings: **Accelerator: GPU T4 x2 or P100**, **Internet: On**.
3. Edit the plan cell below if you like, then **Save Version -> Save & Run All (Commit)**. You can close the laptop.

Each commit stops itself before Kaggle's 12-hour limit and saves `runs/` to the output.
**To continue:** open this notebook again, **Add Input -> Your Work ->** this notebook's latest version
(its `runs/` is picked up and finished runs are skipped), and commit again. Repeat until the summary table is complete.

In [ ]:
# Plan. Stage 1 (main comparison) first, then stage 2 (ablations).
FOLDS = ["europe", "central_asia", "east_asia", "southeast_asia", "south_asia", "africa", "caribbean", "oceania"]
VARIANTS = ["unet", "phydg"]                     # stage 2: ["phydg-nostyle", "phydg-noibn", "phydg-nofilm", "phydg-nopoe", "phydg-novrex"]
SEEDS = [42, 123, 777]                          # stage 2: [42]
EPOCHS = 60
BUDGET_H = 11.0                                 # Kaggle stops commits at 12 h

In [ ]:
import os, subprocess
REPO = "/tmp/repo"
BRANCH = "claude/phydg-slide"
if not os.path.exists(REPO):
    !git clone -q -b {BRANCH} https://github.com/abhishekjadhavcs-afk/abhishekjadhavcs-afk-repo {REPO}
%cd {REPO}
!git pull -q
!pip install -q xarray h5netcdf netCDF4
!df -h /tmp /kaggle/working | cat
!nvidia-smi --query-gpu=name,memory.total --format=csv 2>/dev/null || echo "no GPU (fine for the prepare notebook)"

In [ ]:
import glob, shutil
from pathlib import Path
tars = glob.glob("/kaggle/input/**/cache.tar", recursive=True)
assert tars, "cache.tar not found: add kaggle_1_prepare's output via Add Input"
!tar -xf {tars[0]} -C /tmp && ls /tmp/cache | wc -l

# carry over finished runs from attached previous versions of this notebook
n = 0
for p in glob.glob("/kaggle/input/**/runs/*/*/results.json", recursive=True):
    rel = Path(p).parts[-3:]
    dst = Path("/kaggle/working/runs", *rel)
    if not dst.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(p, dst)
        n += 1
print(f"carried over {n} finished runs")

In [ ]:
!python scripts/run_grid.py --cache /tmp/cache --data-root /tmp/none --out /kaggle/working/runs \
    --folds {" ".join(FOLDS)} --variants {" ".join(VARIANTS)} --seeds {" ".join(map(str, SEEDS))} \
    --time-budget-hours {BUDGET_H} -- --workers 4 --epochs {EPOCHS}

In [ ]:
# checkpoints are not needed for the paper tables; keep the output small
!find /kaggle/working/runs -name best.pt -delete
!cd /kaggle/working && tar -czf results.tar.gz runs && ls -lh results.tar.gz